In [ ]:
import os
import glob
import pandas as pd
import numpy as np

In [ ]:
sparcs_path = "data_csv/SPARCS/"
sparcs_files = glob.glob(os.path.join(sparcs_path, "*.csv"))
asthma_list = []

for file in sparcs_files:
    year = int(os.path.basename(file)[:4])

    chunk_iter = pd.read_csv(file, chunksize=500000, low_memory=False)
    
    for chunk in chunk_iter:
        
        # -------------------------
        # Standardize ZIP column
        # -------------------------
        if "Zip Code - 3 digits" in chunk.columns:
            chunk = chunk.rename(columns={"Zip Code - 3 digits": "zip3"})
        elif "Zip Code" in chunk.columns:
            chunk = chunk.rename(columns={"Zip Code": "zip3"})
        else:
            raise KeyError("No ZIP column found in this file.")
        
        # Clean ZIP3
        chunk["zip3"] = chunk["zip3"].astype(str).str.extract(r'(\d{3})')
        chunk = chunk.dropna(subset=["zip3"])
        
        # Ensure Age Group exists
        if "Age Group" not in chunk.columns:
            raise KeyError("Age Group column not found.")
        
        # -------------------------
        # Standardize diagnosis description column (CCSR vs CCS)
        # -------------------------
        if "CCSR Diagnosis Description" in chunk.columns:
            diag_col = "CCSR Diagnosis Description"
        elif "CCS Diagnosis Description" in chunk.columns:
            diag_col = "CCS Diagnosis Description"
        else:
            raise KeyError(f"No diagnosis description column found in: {basename}")
        
        # -------------------------
        # Filter asthma cases
        # -------------------------
        asthma = chunk[
            chunk[diag_col].str.contains("Asthma", case=False, na=False)
        ].copy()
        
        # Normalize to a single column name for downstream use
        if diag_col != "CCSR Diagnosis Description":
            asthma = asthma.rename(columns={diag_col: "CCSR Diagnosis Description"})
        
        if asthma.empty:
            continue
        
        # -------------------------
        # Clean currency fields (strip $ and commas before converting)
        # -------------------------
        for currency_col in ["Total Charges", "Total Costs"]:
            if currency_col in asthma.columns:
                asthma[currency_col] = (
                    asthma[currency_col]
                    .astype(str)
                    .str.replace(r'[\$,\s]', '', regex=True)
                    .replace('', np.nan)
                    .replace('nan', np.nan)
                    .pipe(pd.to_numeric, errors='coerce')
                )

        # -------------------------
        # Map text-based Risk of Mortality to numeric
        # -------------------------
        mortality_map = {
            "Minor":    1,
            "Moderate": 2,
            "Major":    3,
            "Extreme":  4
        }
        if "APR Risk of Mortality" in asthma.columns:
            asthma["APR Risk of Mortality"] = (
                asthma["APR Risk of Mortality"]
                .astype(str)
                .str.strip()
                .map(mortality_map)
            )

        # -------------------------
        # Clean remaining numeric fields
        # -------------------------
        numeric_cols = [
            "Length of Stay",
            "APR Severity of Illness Code",
            "APR Risk of Mortality",
        ]
        for col in numeric_cols:
            if col in asthma.columns:
                asthma[col] = pd.to_numeric(asthma[col], errors="coerce")
        
        # -------------------------
        # Derived indicators
        # -------------------------
        asthma["ED_Visit"] = np.where(
            asthma["Emergency Department Indicator"].astype(str).str.upper() == "Y", 1, 0
        )
        
        if "Payment Typology 1" in asthma.columns:
            asthma["Medicaid"] = np.where(
                asthma["Payment Typology 1"].str.contains("Medicaid", case=False, na=False),
                1, 0
            )
        else:
            asthma["Medicaid"] = 0
        
        # -------------------------
        # Group by ZIP3 + Age Group
        # -------------------------
        grouped = asthma.groupby(["zip3", "Age Group"]).agg(
            Asthma_Cases=("zip3", "size"),
            ED_Visits=("ED_Visit", "sum"),
            Avg_Length_of_Stay=("Length of Stay", "mean"),
            Avg_Severity=("APR Severity of Illness Code", "mean"),
            Avg_Mortality_Risk=("APR Risk of Mortality", "mean"),
            Total_Charges=("Total Charges", "sum"),
            Total_Costs=("Total Costs", "sum"),
            Avg_Costs=("Total Costs", "mean"),
            Medicaid_Cases=("Medicaid", "sum")
        ).reset_index()
        
        grouped["Year"] = year
        asthma_list.append(grouped)

# ------------------------------------
# Combine all chunks
# ------------------------------------
if not asthma_list:
    print("No asthma cases found across all files.")
else:
    sparcs_combined = pd.concat(asthma_list, ignore_index=True)

    sparcs_final = sparcs_combined.groupby(
        ["Year", "zip3", "Age Group"]
    ).agg({
        "Asthma_Cases":       "sum",
        "ED_Visits":          "sum",
        "Avg_Length_of_Stay": "mean",
        "Avg_Severity":       "mean",
        "Avg_Mortality_Risk": "mean",
        "Total_Charges":      "sum",
        "Total_Costs":        "sum",
        "Avg_Costs":          "mean",
        "Medicaid_Cases":     "sum"
    }).reset_index()

    sparcs_final["ED_Rate"]       = sparcs_final["ED_Visits"]      / sparcs_final["Asthma_Cases"]
    sparcs_final["Medicaid_Rate"] = sparcs_final["Medicaid_Cases"] / sparcs_final["Asthma_Cases"]

    print(sparcs_final.head())

In [ ]:
sparcs_final.to_csv("data_csv/SPARCS_asthma_hospitalizations_by_zip3_2010-2014.csv", index=False)